In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('cleaned_data.csv')
print(data.head())
print("Shape : ",data.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_data.csv'

In [ ]:
from sklearn.model_selection import train_test_split

# Drop non-numeric or irrelevant columns for training, if they exist
# We'll also drop the target variable 'Discounted Price' from features (X)
features = data.drop(columns=['Unnamed: 0', 'Name', 'Discounted Price', 'Model', 'Brand', 'Core'], errors='ignore')

target = data['Discounted Price']

features['Saving_Percentage'] = features['Saving'].str.replace('% OFF', '', regex=False).astype(float)
features = features.drop(columns=['Saving','Actual Price'], errors='ignore')

X = features.select_dtypes(include=np.number)
y = target

print("Features (X) shape before split:", X.shape)
print("Features (X) head after including Saving_Percentage:\n", X.head())
print("Target (y) shape before split:", y.shape)
print("Target (y) head after including Saving_Percentage:\n", y.head())

Features (X) shape before split: (304, 2)
Features (X) head after including Saving_Percentage:
    Actual Price  Saving_Percentage
0      258000.0               21.0
1      350000.0               19.0
2      330000.0               13.0
3      340000.0               15.0
4      175000.0               16.0
Target (y) shape before split: (304,)
Target (y) head after including Saving_Percentage:
 0    203499.0
1    281999.0
2    285999.0
3    287999.0
4    146999.0
Name: Discounted Price, dtype: float64


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("\nShapes after train-test split:")
print(f"Training Set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")


Shapes after train-test split:
Training Set: (212, 2)
Validation set: (46, 2)
Test set: (46, 2)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

results = {
  "Dataset": ["Training", "Validation", "Test"],
  "MSE": [
  mean_squared_error(y_train, y_train_pred),
  mean_squared_error(y_val, y_val_pred),
  mean_squared_error(y_test, y_test_pred)
  ],
  "R2 Score": [
  r2_score(y_train, y_train_pred),
  r2_score(y_val, y_val_pred),
  r2_score(y_test, y_test_pred)
  ]
}
results_df = pd.DataFrame(results)
print(results_df)

      Dataset           MSE  R2 Score
0    Training  1.004419e+08  0.994479
1  Validation  1.345772e+08  0.994762
2        Test  5.906198e+07  0.996737


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mse_scores = -cross_val_score(
  model, X, y, cv=kfold, scoring="neg_mean_squared_error"
)

r2_scores = cross_val_score(
  model, X, y, cv=kfold, scoring="r2"
)

print("MSE for each fold:", mse_scores)
print("Average MSE:", mse_scores.mean())

print("R2 for each fold:", r2_scores)
print("Average R2:", r2_scores.mean())

MSE for each fold: [1.06140546e+08 1.56583434e+08 5.02568041e+07 6.80937329e+07
 1.58170356e+08]
Average MSE: 107848974.86277656
R2 for each fold: [0.99556845 0.99414802 0.9963983  0.99445781 0.99117181]
Average R2: 0.9943488765309987


You can now use `X_train` and `y_train` to train your machine learning model and `X_test` and `y_test` to evaluate its performance.